<a href="https://colab.research.google.com/github/Faucherie/AgileCurrencyProject_midterm/blob/main/cbis_ddsm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# The public DICOM annotations/results Access Routing for TCIA
%pip install idc_index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 132.6 MB/s eta 0:00:00


In [6]:
import os
import sys
import pandas as pd
from idc_index import IDCClient

# 1. Download and Preprocess Data

## 1.1 Download Data

### 1.1.1 Mount Google Drive in Colab

In [7]:
from google.colab import drive

In [8]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# import os

# Create a folder in Drive for the raw files
download_path = '/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw'
os.makedirs(download_path, exist_ok=True)
print(f"Target directory set to: {download_path}")

Target directory set to: /content/drive/MyDrive/CM3070/CBIS_DDSM_Raw


### 1.1.2 Load CSV

Load the ground truth CSVs. These carry the official patient_id-based train/test split.

In [11]:
csv_dir = "/content/drive/MyDrive/CM3070/cbis-csvs"
files = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test":  "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test":  "calc_case_description_test_set.csv",
}


565
3.3G	/content/cbis_ddsm


In [ ]:
#import pandas as pd
#from idc_index import IDCClient

idc_client = IDCClient()

### 1.1.3 Query IDC index for CBIS-DDSM

IDC-index functions return resutls as DFs. With the following header values:

In [ ]:
print(idc_client.index.columns.values)

All the DICOM sereis available from TCIA

In [ ]:
collections = idc_client.index['collection_id'].unique()
analysis_results = idc_client.index['analysis_result_id'].unique()

print("Available Collections:")
display(collections)

print("\nAvailable Analysis Results:")
display(analysis_results)

We want the `'cbis_ddsm'` dataset

In [ ]:
cbis_df = idc_client.index[idc_client.index['collection_id'] == 'cbis_ddsm']
display(cbis_df.head())

Lets summarize the key information for this dataset by `collection_id`

In [ ]:
summary_dict = {
    "collection_id": "cbis_ddsm",
    "Unique_Source_DOIs": ", ".join(cbis_df['source_DOI'].dropna().unique().tolist()),
    "Unique_Licenses": ", ".join(cbis_df['license_short_name'].dropna().unique().tolist()),
    "Unique_Body_Parts": ", ".join(cbis_df['BodyPartExamined'].dropna().unique().tolist()),
    "Unique_Modalities": ", ".join(cbis_df['Modality'].dropna().unique().tolist()),
    "Num_Patients": cbis_df['PatientID'].nunique(),
    "Num_Studies": cbis_df['StudyInstanceUID'].nunique(),
    "Num_Series": cbis_df['SeriesInstanceUID'].nunique(),
    "Total_Series_Size_MB": cbis_df['series_size_MB'].sum(),
  }

display (summary_dict)



As expected the cbis-ddsm examinins breasts as the mamogram `MG` modalitie. There are 6671 patients and 6775 studies as most patients have just one study but a few patients have multiple studies such as differnt views of the same view, multiple abnormalities, or had both breasts examimied.

In [ ]:
help(idc_client.download_from_selection)

In [ ]:
# download full collection 'cbis_ddsm'
idc_client.download_from_selection(collection_id='cbis_ddsm', downloadDir='.')


Let's make this list a little more useful by summarizing other key information contained in idc-index. First, we'll group the relevant data from idc_client.index by collection_id. For each collection, find unique source_DOI and license_short_name values (listed as arrays), count unique PatientID, StudyInstanceUID, and SeriesInstanceUID, and sum series_size_MB. We'll also add a new column named 'Type' with the value 'collection'.

In [ ]:

collections_df = idc_client.index.groupby('collection_id').agg(
    Unique_Source_DOIs=('source_DOI', lambda x: x.dropna().unique().tolist()),
    Unique_Licenses=('license_short_name', lambda x: x.dropna().unique().tolist()),
    Unique_Body_Parts=('BodyPartExamined', lambda x: x.dropna().unique().tolist()),
    Unique_Modalities=('Modality', lambda x: x.dropna().unique().tolist()),
    Num_Patients=('PatientID', 'nunique'),
    Num_Studies=('StudyInstanceUID', 'nunique'),
    Num_Series=('SeriesInstanceUID', 'nunique'),
    Total_Series_Size_MB=('series_size_MB', 'sum')
).reset_index()

collections_df['Type'] = 'collection'

# sort smallest to largest
collections_df = collections_df.sort_values(by='Total_Series_Size_MB', ascending=True)

print("Aggregated Collection Details:")
display(collections_df.head())

Now we'll summarize Analysis Results in the same manner using analysis_result_id.

In [ ]:
analysis_results_df = idc_client.index.groupby('analysis_result_id').agg(
    Unique_Source_DOIs=('source_DOI', lambda x: x.dropna().unique().tolist()),
    Unique_Licenses=('license_short_name', lambda x: x.dropna().unique().tolist()),
    Unique_Body_Parts=('BodyPartExamined', lambda x: x.dropna().unique().tolist()),
    Unique_Modalities=('Modality', lambda x: x.dropna().unique().tolist()),
    Num_Patients=('PatientID', 'nunique'),
    Num_Studies=('StudyInstanceUID', 'nunique'),
    Num_Series=('SeriesInstanceUID', 'nunique'),
    Total_Series_Size_MB=('series_size_MB', 'sum')
).reset_index()

analysis_results_df['Type'] = 'analysis_result'

print("Aggregated Analysis Results Details:")
display(analysis_results_df.head())

And now we can combine collections_df and analysis_results_df in a single dataframe.

In [ ]:

collections_df = collections_df.rename(columns={'collection_id': 'ID'})
analysis_results_df = analysis_results_df.rename(columns={'analysis_result_id': 'ID'})

combined_summary_df = pd.concat([collections_df, analysis_results_df], ignore_index=True)

print("Combined Summary Table (first 5 rows):")
display(combined_summary_df.head())

#### 1.1.3.2 Get a list of availble body parts examined

Next let's see what kinds of values we have in BodyPartExamined. Unfortunately, there is a rather significant volume of data submitted to us where this data is not populated.

In [ ]:
body_part_counts = idc_client.index['BodyPartExamined'].value_counts(dropna=False)

print("DICOM series counts of each unique BodyPartExamined (including NaN):")
display(body_part_counts)

#### 1.1.3.3 Get a list of available modalities

And now let's see what Modality values we have. These abbreviations are explained in more detail in Part 16 Chapter D of the DICOM standard.

In [ ]:
modality_counts = idc_client.index['Modality'].value_counts(dropna=False)

print("DICOM series counts of each unique Modality:")
display(modality_counts)

#### 1.1.3.4 Summarize patient age and sex characteristics

It's important to note that often times this information is missing or may occasionally be filled with non-sense values by the folks who acquired the images. If you can find an accompanying spreadsheet or other source of patient information that should pretty much always be assumed more accurate than this information coming from the DICOM headers, but sometimes it's the only thing available.

In [ ]:
patient_sex_counts = idc_client.index['PatientSex'].value_counts(dropna=False)

print("DICOM series counts of each unique PatientSex:")
display(patient_sex_counts)

Let's look at patient age now. Note that they may be encoded differently with some represented as days (e.g. 399D), months (e.g. 272M) or years (066Y). Again, many of these values are not populated.

In [ ]:
# Get PatientAge data
patient_age = idc_client.index['PatientAge']

# group by age
patient_age = patient_age.value_counts(dropna=False)

display(patient_age)

#### 1.1.3.5 Review metadata for specific DICOM Series

Each row returned by idc-index represents a specific DICOM Series. We are interested in looking at mamograms (`MG`) images of the breast. You might start with something like this:

In [ ]:

modality_counts = idc_client.index['Modality'].value_counts(dropna=False)

print("DICOM series counts of each unique Modality:")
display(modality_counts)

### 1.1.2 Query and Download via TCIA APIInstall

> Add blockquote



the helper library and run the download script. The script fetches the unique series identifiers for CBIS-DDSM and downloads them directly into your Drive folder as zip archives containing the raw DICOM images.

[TCIA API guide](https://www.cancerimagingarchive.net/tcia-api-guides/)

[tcia-utils: a package to simpligy the task of interacting with TCIA](https://pypi.org/project/tcia-utils/)

In [ ]:
!pip install -q tcia-utils

The cbis-ddsm dataset is DICOM radiology data

In [ ]:
from tcia_utils import nbia

Get a list of all series unique identifiers (SeriesInstanceUIDs) for CBIS-DDSM

In [ ]:
print("Fetching series list from TCIA...")
series_list = nbia.getSeries(collection="CBIS-DDSM")
series_uids = [series['SeriesInstanceUID'] for series in series_list]
print(f"Found {len(series_uids)} series to download.")


### Download CBIS-DDSM

In [ ]:
#from idc_index import IDCClient
#import os

idc_client = IDCClient()

# download full collection 'cbis_ddsm'
# 164G total
# Stored in 'CBIS_DDSM_Raw/'
# Remove """" to download
"""
idc_client.download_from_selection(
    collection_id='cbis_ddsm',
    downloadDir=download_path,
    use_s5cmd_sync=True,
    show_progress_bar=True
)
"""


1. Confirm the Drive copy is the complete, authoritative one:

In [21]:
!find /content/drive/MyDrive/CM3070/

Streaming output truncated to the last 5000 lines.
/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw/cbis_ddsm/Mass-Training_P_01889_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.114942472410069947717800792720618051519/MG_1.3.6.1.4.1.9590.100.1.2.383456076011585200833133247132360576495
/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw/cbis_ddsm/Mass-Training_P_01889_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.114942472410069947717800792720618051519/MG_1.3.6.1.4.1.9590.100.1.2.383456076011585200833133247132360576495/9205e559-e32c-4ea6-9bfd-4499bd9b1a1e.dcm
/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw/cbis_ddsm/Mass-Training_P_01889_RIGHT_CC_1
/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw/cbis_ddsm/Mass-Training_P_01889_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.391497925512183671317192780040714946689
/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw/cbis_ddsm/Mass-Training_P_01889_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.391497925512183671317192780040714946689/MG_1.3.6.1.4.1.9590.100.1.2.6527976411897768803524153661443049263
/content/drive/

In [ ]:
  !find /content/drive/MyDrive/CBIS_DDSM_Raw -type f | wc -l
  !du -sh /content/drive/MyDrive/CBIS_DDSM_Raw

2. Compare against the /content/ copy to see if it has anything the Drive copy doesn't (e.g. if the /content run happened first and got further before disconnecting)

In [13]:
print(download_path)

/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw


In [12]:
!find download_path

find: ‘download_path’: No such file or directory
